# Timonel SFT - Banco AObjetivo: entrenar Llama 3.1 8B para responder 1 (terminar) o 2 (ir_a) segun el umbral T.Baseline: Llama base sin SFT = 52.3% (siempre dice ir_a).

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo base

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 1024model, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    load_in_4bit = True,)

## 3. LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.0,    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,)

## 4. Cargar datos

In [ ]:
from datasets import load_datasetTRAIN_PATH = "/kaggle/input/timonel-banco-a/train_A_sft.jsonl"VAL_PATH   = "/kaggle/input/timonel-banco-a/val_A_sft.jsonl"dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")dataset_val = load_dataset("json", data_files=VAL_PATH, split="train")print(f"Train: {len(dataset_train)} | Val: {len(dataset_val)}")

## 5. Chat template

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")def formatear(ejemplo):    return {"text": tokenizer.apply_chat_template(ejemplo["messages"], tokenize=False, add_generation_prompt=False)}dataset_train = dataset_train.map(formatear)dataset_val = dataset_val.map(formatear)print(dataset_train[0]["text"][:500])

## 6. Entrenamiento

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model = model, tokenizer = tokenizer,    train_dataset = dataset_train, eval_dataset = dataset_val,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2, packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 10, max_steps = 150,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 10,        eval_strategy = "steps", eval_steps = 50,        save_strategy = "steps", save_steps = 50,        optim = "adamw_8bit", weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407, output_dir = "outputs_A", report_to = "none",    ),)stats = trainer.train()print(stats)

## 7. Guardar

In [ ]:
model.save_pretrained("timonel_lora_banco_A")tokenizer.save_pretrained("timonel_lora_banco_A")!ls -la timonel_lora_banco_A/